# Evaluación Comparativa Unificada: LMS vs. RLS y Control de Calidad (SQI)

Este notebook unifica la evaluación estadística de **RLS** (Proyecto 2) frente a la familia **LMS** (**LMS estándar** con estructura compartida y **NLMS / VS-LMS** reoptimizados sobre el mismo rango $M \in [6, 50]$ y el pipeline actualizado) sobre `IEEEPPG_TEST.ts`, junto con el análisis de convergencia, costo computacional, ablación de componentes, compromiso de dilatación (acondicionamiento vs. tiempo), sensibilidad y control de calidad de señal (**SQI**).

Carga automáticamente los modelos ganadores desde `optimized_models_config.json` generado al ejecutar `pipeline_optimization_and_evaluation.ipynb`.

| Sección | Contenido | Figuras generadas |
|---|---|---|
| **1** | Carga de datos (`IEEEPPG_TRAIN.ts` / `IEEEPPG_TEST.ts`) y configuración dinámica de modelos | — |
| **2** | Evaluación global en test (detector propio y detector común) vs. línea base del Proyecto 1 | — |
| **3** | Significancia estadística (*bootstrap* por sujeto), CDF del error y comparación pareada | `01_barras_metricas_IC.png`, `03_cdf_y_pareado.png` |
| **4** | Dispersión BPM estimado vs. real y desglose por participante | `02_dispersion_por_metodo.png`, `04_aae_por_sujeto.png` |
| **5** | Casos de estudio en el conjunto de prueba (Ventana #481, mayor mejora y peor caso de RLS) | `06_casos_concretos.png` |
| **6** | Curvas de convergencia en arranque en frío y costo computacional por ventana | `05_convergencia_y_costo.png` |
| **7** | Ablación, compromiso de dilatación a apertura fija, condicionamiento de $\hat{\mathbf{R}}$ y sensibilidad ($N_{\text{eff}}, \delta$) | `07_ablacion.png`, `08_sensibilidad_RLS.png` |
| **8** | Control de calidad de señal (SQI dual AND): calibración, control aleatorio, sensibilidad y regímenes de movimiento | `09_diagnostico_sqi.png`, `10_ejemplos_sqi.png`, `11_regimenes_movimiento_y_ecdf.png` |
| **9** | Resumen cuantitativo final | — |

## 1. Carga de datos y configuración de modelos

In [ ]:
import os, sys, time, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.ndimage import uniform_filter1d
from IPython.display import display

# --- Resolución de rutas del proyecto ---
cwd = os.path.abspath(os.getcwd())
if os.path.basename(cwd) in ("implementation", "restructured_implementation"):
    IMPL_DIR, PROJECT_ROOT = cwd, os.path.abspath(os.path.join(cwd, ".."))
elif os.path.isdir(os.path.join(cwd, "implementation")):
    IMPL_DIR, PROJECT_ROOT = os.path.join(cwd, "implementation"), cwd
else:
    IMPL_DIR, PROJECT_ROOT = cwd, cwd
for p in (IMPL_DIR, PROJECT_ROOT):
    if p not in sys.path:
        sys.path.insert(0, p)

try:
    from implementation import (
        FS, WIN, load_ts, get_participant_groups, preprocess_signal,
        build_U, snr_weighted_late_fusion, zero_padded_periodogram_bpm,
        estimate_cardiac_snr, objective_function, HeartRateEstimatorModel,
        MAX_M, load_optimization_config,
    )
    from implementation import LMS_algorithms as LMSA
except ImportError:
    from dataset_preparation import FS, WIN, load_ts, get_participant_groups
    from preprocessing import preprocess_signal
    from reference_construction import build_U
    from fusion import snr_weighted_late_fusion, estimate_cardiac_snr
    from bpm_detection import zero_padded_periodogram_bpm, objective_function
    from pipeline import HeartRateEstimatorModel
    from optimization import MAX_M, load_optimization_config
    import LMS_algorithms as LMSA

plt.rcParams.update({
    "figure.figsize": (13, 5), "axes.grid": True, "grid.alpha": 0.3,
    "axes.spines.top": False, "axes.spines.right": False, "font.size": 10,
})


def find_dataset_file(filename):
    search_dirs = [
        os.getcwd(), PROJECT_ROOT, IMPL_DIR,
        os.path.join(PROJECT_ROOT, "Proyecto_1_submission"),
        os.path.join(PROJECT_ROOT, "ITBA_PAdSA_Proyecto_2", "implementation"),
        "/content", "/content/drive/MyDrive",
    ]
    for d in search_dirs:
        cand = os.path.join(d, filename)
        if os.path.isfile(cand):
            return cand
    raise FileNotFoundError(f"No se encontró '{filename}' en {search_dirs}")


X_train, y_train = load_ts(find_dataset_file("IEEEPPG_TRAIN.ts"))
X_test, y_test = load_ts(find_dataset_file("IEEEPPG_TEST.ts"))
train_groups = get_participant_groups(X_train)
test_groups = get_participant_groups(X_test)
subjects = np.unique(test_groups)

print(f"Train: {X_train.shape} | sujetos = {len(np.unique(train_groups))}")
print(f"Test : {X_test.shape} | sujetos = {len(subjects)}")

In [ ]:
# ----------------------------- Configuración General -----------------------------
SEED = 0
SAVE_FIGS = True
FIG_DIR = "figuras_comparacion_unificada"
LMS_REF = "nlms"
N_BOOT = 2000
N_CURVE = 150
RUN_ABLATION = True
RUN_SENSITIVITY = True
ABL_N = 300
OUTLIER_BPM = 10.0

SQI_PCT_ACC = 75
SQI_PCT_SNR = 25
N_RANDOM_CTRL = 1000

os.makedirs(FIG_DIR, exist_ok=True)


def save_fig(fig, name):
    if SAVE_FIGS:
        fig.savefig(os.path.join(FIG_DIR, f"{name}.png"), dpi=200, bbox_inches="tight")


def find_optional_config(filename="optimized_models_config.json"):
    search_dirs = [
        IMPL_DIR, PROJECT_ROOT, os.getcwd(),
        os.path.join(PROJECT_ROOT, "ITBA_PAdSA_Proyecto_2", "implementation"),
        "/content", "/content/drive/MyDrive",
    ]
    for d in search_dirs:
        cand = os.path.join(d, filename)
        if os.path.isfile(cand):
            return cand
    return None


# ----------------------------- Modelos comparados --------------------------------
cfg_path = find_optional_config()
if cfg_path is not None:
    loaded_cfg = load_optimization_config(cfg_path)
    algos_cfg = loaded_cfg["algorithms"]
    RLS_M = int(algos_cfg["rls"]["M"])
    RLS_PARAMS = dict(algos_cfg["rls"]["lms_params"])
    RLS_HR = dict(hr_algorithm=algos_cfg["rls"]["hr_algorithm"], hr_params=dict(algos_cfg["rls"]["hr_params"]))

    NLMS_M = int(algos_cfg["nlms"]["M"])
    NLMS_PARAMS = dict(algos_cfg["nlms"]["lms_params"])
    NLMS_HR = dict(hr_algorithm=algos_cfg["nlms"]["hr_algorithm"], hr_params=dict(algos_cfg["nlms"]["hr_params"]))

    VS_LMS_M = int(algos_cfg["vs_lms"]["M"])
    VS_LMS_PARAMS = dict(algos_cfg["vs_lms"]["lms_params"])
    VS_LMS_HR = dict(hr_algorithm=algos_cfg["vs_lms"]["hr_algorithm"], hr_params=dict(algos_cfg["vs_lms"]["hr_params"]))

    CV_AAE = {
        "base": float(loaded_cfg.get("baseline_cv", {}).get("score_aae", 12.52)),
        "lms": 5.18,
        "nlms": float(algos_cfg["nlms"]["cv"]["score_aae"]),
        "vs_lms": float(algos_cfg["vs_lms"]["cv"]["score_aae"]),
        "rls": float(algos_cfg["rls"]["cv"]["score_aae"]),
    }
    print(f"[CONFIG] Cargado desde {cfg_path} (MAX_M={loaded_cfg.get('max_m_searched', MAX_M)})")
    if loaded_cfg.get("boundary_warnings"):
        print("[WARNING] Advertencias de límite:", loaded_cfg["boundary_warnings"])
else:
    print("[AVISO] No se encontró optimized_models_config.json; usando valores de referencia previos.")
    RLS_M = 7
    RLS_PARAMS = dict(
        delay_step=4,
        N_eff=1409.281222670354,
        delta=5.11410341560248,
        diag_load=0.14339015026730578,
        huber_c=2.5014879715311853,
        warm_start_ratio=0.0,
    )
    RLS_HR = dict(hr_algorithm="periodogram", hr_params=dict(nfft=8192, window_type="hann"))

    NLMS_M = 16
    NLMS_PARAMS = dict(
        delay_step=2,
        mu=0.01459,
        delta=2.63535,
        huber_c=1.68220,
        warm_start_ratio=0.5,
    )
    NLMS_HR = dict(hr_algorithm="periodogram", hr_params=dict(nfft=8192, window_type="hamming"))

    VS_LMS_M = 13
    VS_LMS_PARAMS = dict(
        delay_step=2,
        mu_max=0.15297,
        mu_min=0.01317,
        alpha=0.97338,
        gamma=0.00012,
        delta=0.79051,
        huber_c=1.80559,
        warm_start_ratio=0.5,
    )
    VS_LMS_HR = dict(hr_algorithm="periodogram", hr_params=dict(nfft=8192, window_type="hamming"))
    CV_AAE = {"base": 12.52, "lms": 5.18, "nlms": 3.82, "vs_lms": 4.04, "rls": 3.66}

print(f"  RLS    : M={RLS_M} (L={3*RLS_M}), D={RLS_PARAMS['delay_step']}, delta={RLS_PARAMS['delta']:.4f}, N_eff={RLS_PARAMS.get('N_eff', 'lam')}, dl={RLS_PARAMS['diag_load']:.4f}, c={RLS_PARAMS['huber_c']:.3f}, WS={RLS_PARAMS['warm_start_ratio']}")
print(f"  NLMS   : M={NLMS_M} (L={3*NLMS_M}), D={NLMS_PARAMS['delay_step']}, mu={NLMS_PARAMS['mu']:.5f}, delta={NLMS_PARAMS['delta']:.4f}, c={NLMS_PARAMS['huber_c']:.3f}, WS={NLMS_PARAMS['warm_start_ratio']}")
print(f"  VS-LMS : M={VS_LMS_M} (L={3*VS_LMS_M}), D={VS_LMS_PARAMS['delay_step']}, c={VS_LMS_PARAMS['huber_c']:.3f}, WS={VS_LMS_PARAMS['warm_start_ratio']}")

# LMS estándar conserva su paso óptimo del Proyecto 1 con la estructura de RLS
SHARED_STRUCTURAL_PARAMS = dict(
    delay_step=RLS_PARAMS["delay_step"],
    huber_c=RLS_PARAMS["huber_c"],
    warm_start_ratio=RLS_PARAMS["warm_start_ratio"],
)

models = {
    "lms": HeartRateEstimatorModel.from_project1_optimal("lms", **SHARED_STRUCTURAL_PARAMS),
    "nlms": HeartRateEstimatorModel(
        lms_algorithm="nlms", M=NLMS_M, lms_params=dict(NLMS_PARAMS), **NLMS_HR),
    "vs_lms": HeartRateEstimatorModel(
        lms_algorithm="vs_lms", M=VS_LMS_M, lms_params=dict(VS_LMS_PARAMS), **VS_LMS_HR),
    "rls": HeartRateEstimatorModel(
        lms_algorithm="rls", M=RLS_M, lms_params=dict(RLS_PARAMS), **RLS_HR),
}

REF_TEST_AAE = {"base": 20.41, "lms": 16.01, "nlms": 14.05, "vs_lms": 15.76, "rls": 12.23}

ORDER = ["base", "lms", "nlms", "vs_lms", "rls"]
LABELS = {"base": "Sin filtro", "lms": "LMS", "nlms": "NLMS", "vs_lms": "VS-LMS", "rls": "RLS"}
COLORS = {"base": "#7f7f7f", "lms": "#9467bd", "nlms": "#1f77b4", "vs_lms": "#2ca02c", "rls": "#d62728"}
assert LMS_REF in models

## 2. Evaluación global sobre el conjunto de prueba

Cada ventana de $8\text{ s}$ se procesa de forma independiente. Se reporta el rendimiento con el **detector óptimo propio** de cada algoritmo y con un **detector común** (periodograma Hann, $\text{NFFT}=8192$, idéntico al de RLS; y Hamming, $\text{NFFT}=4096$ como control adicional) para aislar el efecto propio del filtro adaptativo.

In [ ]:
def compute_metrics(y_true, y_pred, mask=None):
    if mask is not None:
        y_true, y_pred = y_true[mask], y_pred[mask]
    if len(y_true) == 0:
        return {"N": 0, "aae": np.nan, "medae": np.nan, "rmse": np.nan, "p90": np.nan, "outlier_pct": np.nan, "obj": np.nan}
    err = np.abs(y_true - y_pred)
    return {
        "N": int(len(y_true)),
        "aae": float(np.mean(err)),
        "medae": float(np.median(err)),
        "rmse": float(np.sqrt(np.mean(err ** 2))),
        "p90": float(np.percentile(err, 90)),
        "outlier_pct": float(100.0 * np.mean(err > OUTLIER_BPM)),
        "obj": float(objective_function(y_pred, y_true)),
    }


def fill_nan(p, fallback):
    p = np.asarray(p, dtype=float).copy()
    bad = ~np.isfinite(p)
    p[bad] = fallback[bad]
    return p, int(bad.sum())


def safe_mean(v):
    v = np.asarray(v, dtype=float)
    return float(np.mean(v)) if v.size else np.nan


COMMON_HANN = dict(nfft=8192, window_type="hann")
COMMON_HAMM = dict(nfft=4096, window_type="hamming")

ppg1_t, ppg2_t, acc_t = preprocess_signal(X_test)
d_fused_t = np.array([snr_weighted_late_fusion(ppg1_t[i], ppg2_t[i]) for i in range(len(X_test))])
pred_base_hann = np.array([zero_padded_periodogram_bpm(s, **COMMON_HANN) for s in d_fused_t])
pred_base_hamm = np.array([zero_padded_periodogram_bpm(s, **COMMON_HAMM) for s in d_fused_t])

test_eval = {
    "base": {
        "preds": pred_base_hann,
        "preds_common": pred_base_hann,
        "preds_hamm": pred_base_hamm,
        "signals": d_fused_t,
        "elapsed": 0.0,
        "n_nan": 0,
    }
}

for key, model in models.items():
    t0 = time.perf_counter()
    preds, sigs = model.predict_with_signals(X_test)
    elapsed_s = time.perf_counter() - t0
    p_hann = np.array([zero_padded_periodogram_bpm(s, **COMMON_HANN) for s in sigs])
    p_hamm = np.array([zero_padded_periodogram_bpm(s, **COMMON_HAMM) for s in sigs])
    preds, n1 = fill_nan(preds, pred_base_hann)
    p_hann, n2 = fill_nan(p_hann, pred_base_hann)
    p_hamm, n3 = fill_nan(p_hamm, pred_base_hamm)
    test_eval[key] = {
        "preds": preds,
        "preds_common": p_hann,
        "preds_hamm": p_hamm,
        "signals": sigs,
        "elapsed": elapsed_s,
        "n_nan": max(n1, n2, n3),
    }
    print(f"{LABELS[key]:<7} listo en {elapsed_s:6.2f} s  (ventanas con NaN reemplazadas: {max(n1, n2, n3)})")

for key, ev in test_eval.items():
    ev.update(compute_metrics(y_test, ev["preds"]))
    ev["abs_err"] = np.abs(ev["preds"] - y_test)
    ev["abs_err_common"] = np.abs(ev["preds_common"] - y_test)
    ev["common"] = compute_metrics(y_test, ev["preds_common"])
    ev["hamm"] = compute_metrics(y_test, ev["preds_hamm"])

rows = []
for k in ORDER:
    ev = test_eval[k]
    rows.append({
        "Método": LABELS[k],
        "AAE val. (CV)": CV_AAE[k],
        "AAE test": ev["aae"],
        "RMSE test": ev["rmse"],
        "MedAE test": ev["medae"],
        "Outliers (%)": ev["outlier_pct"],
        "AAE (Hann 8192)": ev["common"]["aae"],
        "AAE (Hamm 4096)": ev["hamm"]["aae"],
        "Tiempo (s)": ev["elapsed"],
    })
df_resumen = pd.DataFrame(rows).set_index("Método")
display(df_resumen.round(2))

print("\nComparación frente al Proyecto 1 (D=1, sin Huber):")
for k in ORDER:
    diff = test_eval[k]["aae"] - REF_TEST_AAE[k]
    print(f"  {LABELS[k]:<10} Proy.1={REF_TEST_AAE[k]:6.2f} | Actual={test_eval[k]['aae']:6.2f} | dif={diff:+.2f} BPM")

## 3. Significancia estadística (*bootstrap* por sujeto), CDF y comparación pareada

Las ventanas consecutivas de un mismo participante comparten $6\text{ s}$ de solapamiento, por lo que remuestreamos **participantes completos** ($N_{\text{boot}}=2000$ réplicas). A continuación se muestran la CDF del error absoluto y la comparación pareada ventana a ventana entre RLS y NLMS.

In [ ]:
def cluster_bootstrap_ci(values, groups, stat=np.mean, n_boot=N_BOOT, alpha=0.05, seed=SEED):
    rng = np.random.default_rng(seed)
    ug = np.unique(groups)
    parts = [values[groups == g] for g in ug]
    out = np.empty(n_boot)
    for b in range(n_boot):
        sel = rng.integers(0, len(ug), len(ug))
        out[b] = stat(np.concatenate([parts[j] for j in sel]))
    return float(stat(values)), np.percentile(out, [100 * alpha / 2, 100 * (1 - alpha / 2)])


def subject_bootstrap_diff_ci(y_true, pred_a, pred_b, groups, mask=None, n_boot=N_BOOT, seed=SEED):
    rng = np.random.default_rng(seed)
    if mask is not None:
        y_true, pred_a, pred_b, groups = y_true[mask], pred_a[mask], pred_b[mask], groups[mask]
    uniq = np.unique(groups)
    err_a, err_b = np.abs(y_true - pred_a), np.abs(y_true - pred_b)
    idx_by_s = [np.where(groups == s)[0] for s in uniq]
    diffs = np.empty(n_boot)
    for b in range(n_boot):
        pick = rng.integers(0, len(uniq), size=len(uniq))
        idx = np.concatenate([idx_by_s[p] for p in pick])
        diffs[b] = err_a[idx].mean() - err_b[idx].mean()
    return float(err_a.mean() - err_b.mean()), float(diffs.mean()), float(np.percentile(diffs, 2.5)), float(np.percentile(diffs, 97.5))


stats = [
    ("AAE (BPM)", np.mean),
    ("Error mediano (BPM)", np.median),
    (f"Outliers (% > {OUTLIER_BPM:g} BPM)", lambda v: 100.0 * np.mean(v > OUTLIER_BPM)),
]
fig, axes = plt.subplots(1, 3, figsize=(15, 4.3))
for ax, (title, stat) in zip(axes, stats):
    for j, k in enumerate(ORDER):
        val, (lo, hi) = cluster_bootstrap_ci(test_eval[k]["abs_err"], test_groups, stat)
        ax.bar(j, val, color=COLORS[k], yerr=[[val - lo], [hi - val]], capsize=4, alpha=0.9)
        ax.text(j, hi, f"{val:.2f}", ha="center", va="bottom", fontsize=9)
    ax.set_xticks(range(len(ORDER)))
    ax.set_xticklabels([LABELS[k] for k in ORDER])
    ax.set_title(title)
    ax.grid(axis="x", visible=False)
fig.suptitle("Rendimiento en test (IC 95 % por bootstrap sobre sujetos)", y=1.02)
fig.tight_layout()
save_fig(fig, "01_barras_metricas_IC")
plt.show()

print("Diferencia pareada de AAE (positiva = RLS mejor), IC 95 % por sujeto (global):")
for k in ["base", "lms", "nlms", "vs_lms"]:
    pt, mb, lo, hi = subject_bootstrap_diff_ci(y_test, test_eval[k]["preds"], test_eval["rls"]["preds"], test_groups)
    verdict = "el IC excluye 0 -> RLS mejor" if lo > 0 else ("RLS peor" if hi < 0 else "el IC incluye 0 -> no concluyente")
    win_pct = np.mean((test_eval[k]["abs_err"] - test_eval["rls"]["abs_err"]) > 0) * 100
    print(f"  {LABELS[k]:<10} - RLS : punt={pt:+6.2f} | boot={mb:+6.2f} BPM  [{lo:+.2f}, {hi:+.2f}] ({verdict}; gana en {win_pct:4.1f}%)")

fig, axes = plt.subplots(1, 2, figsize=(15, 5.2))
ax = axes[0]
for k in ORDER:
    e = np.sort(test_eval[k]["abs_err"])
    ax.plot(np.maximum(e, 0.05), np.arange(1, len(e) + 1) / len(e), color=COLORS[k], lw=2.2 if k == "rls" else 1.6,
            label=f"{LABELS[k]} ({100 - test_eval[k]['outlier_pct']:.1f}% ≤ {OUTLIER_BPM:g} BPM)")
ax.axvline(OUTLIER_BPM, color="k", ls="--", lw=1)
ax.set_xscale("log")
ax.set_xlabel("Error absoluto |BPM est. − BPM real|  (escala log)")
ax.set_ylabel("Fracción acumulada de ventanas")
ax.set_title("CDF del error absoluto en test")
ax.legend(loc="lower right", fontsize=9)

ax = axes[1]
e_ref = np.maximum(test_eval[LMS_REF]["abs_err"], 0.05)
e_rls = np.maximum(test_eval["rls"]["abs_err"], 0.05)
better = test_eval["rls"]["abs_err"] < test_eval[LMS_REF]["abs_err"] - 1.0
worse = test_eval["rls"]["abs_err"] > test_eval[LMS_REF]["abs_err"] + 1.0
same = ~better & ~worse
ax.scatter(e_ref[same], e_rls[same], s=10, color="#bbbbbb", label=f"Similar (±1 BPM): {same.sum()}")
ax.scatter(e_ref[better], e_rls[better], s=12, color=COLORS["rls"], alpha=0.7, label=f"RLS mejor: {better.sum()}")
ax.scatter(e_ref[worse], e_rls[worse], s=12, color=COLORS[LMS_REF], alpha=0.7, label=f"{LABELS[LMS_REF]} mejor: {worse.sum()}")
lims = (0.04, 300)
ax.plot(lims, lims, "k--", lw=1)
ax.axvline(OUTLIER_BPM, color="k", ls=":", lw=0.8); ax.axhline(OUTLIER_BPM, color="k", ls=":", lw=0.8)
ax.set_xscale("log"); ax.set_yscale("log"); ax.set_xlim(lims); ax.set_ylim(lims)
ax.set_xlabel(f"Error {LABELS[LMS_REF]} (BPM)")
ax.set_ylabel("Error RLS (BPM)")
ax.set_title(f"Comparación pareada por ventana: RLS vs. {LABELS[LMS_REF]}")
ax.legend(loc="upper left", fontsize=9)
fig.tight_layout()
save_fig(fig, "03_cdf_y_pareado")
plt.show()

## 4. Dispersión BPM estimado vs. real y rendimiento por participante

In [ ]:
cmap = plt.get_cmap("tab10")
lim = (25, 240)
fig, axes = plt.subplots(2, 3, figsize=(16, 10), sharex=True, sharey=True)
axf = axes.ravel()
for ax, k in zip(axf, ORDER):
    for s in subjects:
        idx = test_groups == s
        ax.scatter(y_test[idx], test_eval[k]["preds"][idx], s=9, alpha=0.6, color=cmap(s % 10), label=f"Suj. {s + 1}")
    xs = np.array(lim)
    ax.plot(xs, xs, "k--", lw=1)
    ax.fill_between(xs, xs - OUTLIER_BPM, xs + OUTLIER_BPM, color="crimson", alpha=0.10)
    ax.set_xlim(lim); ax.set_ylim(lim)
    ax.set_title(f"{LABELS[k]}  (AAE = {test_eval[k]['aae']:.2f}, outliers = {test_eval[k]['outlier_pct']:.1f}%)")
    ax.set_xlabel("BPM real"); ax.set_ylabel("BPM estimado")
h, l = axf[0].get_legend_handles_labels()
axf[-1].axis("off")
axf[-1].legend(h, l, loc="center", ncol=2, title="Sujeto de test", frameon=False, markerscale=2)
fig.suptitle("Test: BPM estimado vs. real por método", y=0.995)
fig.tight_layout()
save_fig(fig, "02_dispersion_por_metodo")
plt.show()

aae_subj = pd.DataFrame(
    {LABELS[k]: [test_eval[k]["abs_err"][test_groups == s].mean() for s in subjects] for k in ORDER},
    index=[f"Sujeto {s + 1}" for s in subjects],
)
aae_subj.insert(0, "Ventanas", [int(np.sum(test_groups == s)) for s in subjects])
display(aae_subj.round(2))

fig, axes = plt.subplots(1, 2, figsize=(16, 4.8), gridspec_kw={"width_ratios": [2.2, 1]})
ax = axes[0]
cmp_keys = ["lms", "nlms", "vs_lms", "rls"]
wbar = 0.2
for j, k in enumerate(cmp_keys):
    ax.bar(subjects + 1 + (j - 1.5) * wbar, aae_subj[LABELS[k]].values, width=wbar, color=COLORS[k], label=LABELS[k])
ax.set_xticks(subjects + 1)
ax.set_xlabel("Sujeto de test"); ax.set_ylabel("AAE (BPM)"); ax.set_title("AAE por sujeto"); ax.legend(ncol=4, fontsize=9)

ax = axes[1]
gain = aae_subj[LABELS[LMS_REF]].values - aae_subj["RLS"].values
ax.bar(subjects + 1, gain, color=np.where(gain > 0, "#2ca02c", "#d62728"))
ax.axhline(0, color="k", lw=1)
ax.set_xticks(subjects + 1)
ax.set_xlabel("Sujeto de test"); ax.set_ylabel(f"AAE {LABELS[LMS_REF]} − AAE RLS (BPM)")
ax.set_title(f"Mejora de RLS sobre {LABELS[LMS_REF]}")
fig.tight_layout()
save_fig(fig, "04_aae_por_sujeto")
plt.show()
print(f"RLS mejora a {LABELS[LMS_REF]} en {int(np.sum(gain > 0))} de {len(subjects)} sujetos.")

## 5. Casos de estudio en el conjunto de prueba

In [ ]:
t_axis = np.arange(WIN) / FS

def spectrum_bpm(x, nfft=8192):
    x = np.asarray(x, dtype=float)
    if not np.all(np.isfinite(x)):
        return None, None
    x = x - x.mean()
    X = np.abs(np.fft.rfft(x * np.hanning(len(x)), n=nfft))
    f = np.fft.rfftfreq(nfft, 1.0 / FS)
    m = (f >= 0.5) & (f <= 4.0)
    return 60 * f[m], X[m] / (X[m].max() + 1e-12)


gain_win = test_eval[LMS_REF]["abs_err_common"] - test_eval["rls"]["abs_err_common"]
cases = [
    ("Ventana #481 (outlier del Proyecto 1)", 481),
    (f"Mayor mejora de RLS sobre {LABELS[LMS_REF]}", int(np.argmax(gain_win))),
    (f"Mayor empeoramiento de RLS respecto de {LABELS[LMS_REF]}", int(np.argmin(gain_win))),
]

show = [("base", "Sin filtro"), (LMS_REF, LABELS[LMS_REF]), ("rls", "RLS")]
fig, axes = plt.subplots(len(cases), 2, figsize=(16, 3.9 * len(cases)), squeeze=False)
for r, (title, idx) in enumerate(cases):
    axt, axs = axes[r]
    for j, (k, lab) in enumerate(show):
        sig = test_eval[k]["signals"][idx]
        if np.all(np.isfinite(sig)):
            axt.plot(t_axis, (sig - sig.mean()) / (sig.std() + 1e-12) - 4.2 * j, color=COLORS[k], lw=1.3, label=lab)
        f, S = spectrum_bpm(sig)
        if f is not None:
            axs.plot(f, S, color=COLORS[k], lw=1.5, label=f"{lab}: {test_eval[k]['preds_common'][idx]:.1f} BPM")
            axs.plot(test_eval[k]['preds_common'][idx], 1.03, marker="v", color=COLORS[k], ms=9, clip_on=False)
    axs.axvline(y_test[idx], color="k", lw=1.5, ls="--", label=f"Real: {y_test[idx]:.1f} BPM")
    axt.set_title(f"{title} | ventana #{idx} (sujeto {test_groups[idx] + 1})")
    axt.set_yticks([]); axt.set_xlabel("Tiempo (s)"); axt.legend(loc="upper right", fontsize=8, ncol=3)
    axs.set_title("Espectro (periodograma Hann, nfft = 8192)")
    axs.set_xlabel("BPM"); axs.set_ylabel("Magnitud normalizada"); axs.set_xlim(30, 240); axs.legend(loc="upper right", fontsize=8)
    print(f"Caso #{idx} (Sujeto {test_groups[idx]+1}, Real={y_test[idx]:.1f}): " +
          ", ".join(f"{lab}={test_eval[k]['preds_common'][idx]:.1f}" for k, lab in show))
fig.tight_layout()
save_fig(fig, "06_casos_concretos")
plt.show()

## 6. Convergencia dentro de la ventana y costo computacional

In [ ]:
rng = np.random.default_rng(SEED)
curve_idx = rng.choice(len(X_test), size=min(N_CURVE, len(X_test)), replace=False)
pp1, pp2, pacc = preprocess_signal(X_test[curve_idx])

curves, times = {}, {}
for k, model in models.items():
    P, tt = [], []
    for i in range(len(curve_idx)):
        t0 = time.perf_counter()
        _, det = LMSA.lms_bucket(pp1[i], pp2[i], pacc[i], algorithm=model.lms_algorithm, M=model.M,
                                 return_details=True, **model.lms_params)
        tt.append(time.perf_counter() - t0)
        P.append(0.5 * (det["e1"] ** 2 + det["e2"] ** 2))
    curves[k] = np.array(P)
    times[k] = 1e3 * np.median(tt)

fig, axes = plt.subplots(1, 3, figsize=(17, 4.8), gridspec_kw={"width_ratios": [1.4, 1.4, 1]})
for ax, tmax in zip(axes[:2], (WIN / FS, 2.0)):
    for k in ["lms", "nlms", "vs_lms", "rls"]:
        med = np.nanmedian(curves[k], axis=0)
        sm = uniform_filter1d(med, size=25, mode="nearest")
        ax.plot(t_axis, 10 * np.log10(sm + 1e-12), color=COLORS[k], lw=2.2 if k == "rls" else 1.6, label=LABELS[k])
    ax.set_xlim(0, tmax); ax.set_xlabel("Tiempo dentro de la ventana (s)"); ax.set_ylabel("Potencia del residuo (dB re. entrada)")
axes[0].set_title("Convergencia (ventana completa)"); axes[1].set_title("Zoom: primeros 2 s (arranque en frío)"); axes[0].legend()

ax = axes[2]
ks = ["lms", "nlms", "vs_lms", "rls"]
Ls = {k: 3 * models[k].M for k in ks}
ax.bar(range(len(ks)), [times[k] for k in ks], color=[COLORS[k] for k in ks])
for j, k in enumerate(ks):
    ax.text(j, times[k], f"{times[k]:.1f} ms\nL={Ls[k]}", ha="center", va="bottom", fontsize=9)
ax.set_xticks(range(len(ks))); ax.set_xticklabels([LABELS[k] for k in ks])
ax.set_ylabel("ms por ventana (2 canales + fusión)"); ax.set_title("Tiempo de cómputo (mediana)")
ax.set_ylim(0, max(times.values()) * 1.3); ax.grid(axis="x", visible=False)
fig.tight_layout()
save_fig(fig, "05_convergencia_y_costo")
plt.show()
print("Mediana ms/ventana:", {LABELS[k]: round(times[k], 1) for k in ks})
print("Costo relativo respecto de NLMS:", {LABELS[k]: round(times[k] / times["nlms"], 2) for k in ks})

## 7. Ablación de componentes, compromiso de dilatación, condicionamiento y sensibilidad de RLS

Evaluamos cada componente sobre un subconjunto fijo de $300$ ventanas de **entrenamiento** con detector común (periodograma Hann, $\text{NFFT}=8192$). Además:
- Cuantificamos el **compromiso de dilatación a apertura temporal constante** ($D \in \{1, 2, 3, 4\}$), evaluando la dilatación como mecanismo posible de condicionamiento/regularización y ahorro de tiempo de cómputo ($\mathcal{O}(L^2)$).
- Verificamos el espectro de $\hat{\mathbf{R}}=\mathbf{U}^T\mathbf{U}$ para explicar la escala física de la regularización inicial $\delta_0=\delta L$ y extendemos el barrido de $\delta$ hasta $\delta=500$.

In [ ]:
def prepare_subset(X, n, seed=SEED):
    r = np.random.default_rng(seed)
    idx = np.sort(r.choice(len(X), size=min(n, len(X)), replace=False))
    p1, p2, ac = preprocess_signal(X[idx])
    d_f = np.array([snr_weighted_late_fusion(p1[i], p2[i]) for i in range(len(idx))])
    base = np.array([zero_padded_periodogram_bpm(s, nfft=8192) for s in d_f])
    return idx, p1, p2, ac, base


def eval_filter(algorithm, M, params, subset, y):
    _, p1, p2, ac, base = subset
    preds = np.empty(len(base))
    for i in range(len(base)):
        e = LMSA.lms_bucket(p1[i], p2[i], ac[i], algorithm=algorithm, M=M, return_details=False, **params)
        preds[i] = zero_padded_periodogram_bpm(e, nfft=8192, window_type="hann")
    preds, _ = fill_nan(preds, base)
    err = np.abs(preds - y)
    return dict(aae=float(err.mean()), medae=float(np.median(err)), out=float(100 * np.mean(err > OUTLIER_BPM)))


RLS_D = int(RLS_PARAMS["delay_step"])
NLMS_D = int(NLMS_PARAMS["delay_step"])
VS_LMS_D = int(VS_LMS_PARAMS["delay_step"])
T_AP_SAMPLES = max(6, (RLS_M - 1) * RLS_D)
M_NODIL = max(6, min(MAX_M, T_AP_SAMPLES + 1))

if RUN_ABLATION or RUN_SENSITIVITY:
    subset = prepare_subset(X_train, ABL_N)
    y_sub = y_train[subset[0]]
    base_err = np.abs(subset[4] - y_sub)
    print(f"Subconjunto de entrenamiento: {len(y_sub)} ventanas | AAE sin filtro = {base_err.mean():.2f} BPM")

    # Espectro de R_hat = U^T U en este subconjunto
    _, _, _, ac_sub, _ = subset
    cond_dense_15, cond_opt, lmax_opt, tr_opt = [], [], [], []
    for a in ac_sub[:100]:
        U1 = build_U(a, M=15, delay_step=1); e1 = np.linalg.eigvalsh(U1.T @ U1)
        Uo = build_U(a, M=RLS_M, delay_step=RLS_D); eo = np.linalg.eigvalsh(Uo.T @ Uo)
        cond_dense_15.append(e1[-1] / max(e1[0], 1e-12))
        cond_opt.append(eo[-1] / max(eo[0], 1e-12))
        lmax_opt.append(eo[-1]); tr_opt.append(eo.sum())
    L_rls = 3 * RLS_M
    delta_opt = float(RLS_PARAMS["delta"])
    print(f"Condicionamiento mediano cond(U^T U): (D=1,M=15)={np.median(cond_dense_15):.2e} | RLS opt (D={RLS_D},M={RLS_M})={np.median(cond_opt):.2e}")
    print(f"Para RLS opt (D={RLS_D}, M={RLS_M}, L={L_rls}): mediana lambda_max(R_hat)={np.median(lmax_opt):.1f}, traza={np.median(tr_opt):.1f}; delta_0 = {delta_opt:.3f}*{L_rls} = {delta_opt*L_rls:.1f} ({100.0*delta_opt*L_rls/np.median(lmax_opt):.2f}% de lambda_max)")

if RUN_ABLATION:
    nlms_p = {"mu": 0.01288, "delta": 0.5}
    vs_p = {k: v for k, v in LMSA.OPTIMAL_VS_LMS_PARAMS.items() if k != "M"}
    rls_full = dict(RLS_PARAMS)

    def rls_with(**over):
        p = dict(rls_full)
        for k, v in over.items():
            if v is None: p.pop(k, None)
            else: p[k] = v
        return p

    variants = [
        ("LMS Proyecto 1 (D=1, M=11)",                               "lms",    "lms",    11,       {"mu": 0.00070}),
        (f"LMS + estructura compartida (D={RLS_D}, M=11)",           "lms",    "lms",    11,       {"mu": 0.00070, **SHARED_STRUCTURAL_PARAMS}),
        ("NLMS Proyecto 1 (D=1, M=22)",                              "lms",    "nlms",   22,       nlms_p),
        (f"NLMS con estructura RLS (D={RLS_D}, M={RLS_M})",          "lms",    "nlms",   RLS_M,    {**nlms_p, **SHARED_STRUCTURAL_PARAMS}),
        (f"NLMS reoptimizado P2 (D={NLMS_D}, M={NLMS_M})",           "lms",    "nlms",   NLMS_M,   dict(NLMS_PARAMS)),
        ("VS-LMS Proyecto 1 (D=1, M=18)",                            "lms",    "vs_lms", 18,       vs_p),
        (f"VS-LMS con estructura RLS (D={RLS_D}, M={RLS_M})",        "lms",    "vs_lms", RLS_M,    {**vs_p, **SHARED_STRUCTURAL_PARAMS}),
        (f"VS-LMS reoptimizado P2 (D={VS_LMS_D}, M={VS_LMS_M})",     "lms",    "vs_lms", VS_LMS_M, dict(VS_LMS_PARAMS)),
        ("RLS ingenuo (D=1, M=15, λ=0.98, δ=0.01)",                  "rls",    "rls",    15,       dict(delay_step=1, lam=0.98, delta=0.01, diag_load=0.0, huber_c=0.0, warm_start_ratio=0.0)),
        (f"RLS final (D={RLS_D}, M={RLS_M})",                        "rls",    "rls",    RLS_M,    rls_full),
    ]
    if RLS_D > 1:
        variants.append((f"RLS final sin dilatación (D=1, M={M_NODIL})", "rls", "rls", M_NODIL, rls_with(delay_step=1)))
    else:
        for d_alt in (2, 4):
            m_alt = max(6, min(MAX_M, int(round(T_AP_SAMPLES / d_alt)) + 1))
            variants.append((f"RLS final con dilatación (D={d_alt}, M={m_alt})", "rls", "rls", m_alt, rls_with(delay_step=d_alt)))

    variants.extend([
        ("RLS final sin Huber",                        "rls",    "rls",    RLS_M,    rls_with(huber_c=0.0)),
        ("RLS final sin carga diagonal",               "rls",    "rls",    RLS_M,    rls_with(diag_load=0.0)),
        ("RLS final sin olvido (λ=1)",                 "rls",    "rls",    RLS_M,    rls_with(N_eff=None, lam=1.0)),
        ("RLS final + warm start (35 %)",              "rls",    "rls",    RLS_M,    rls_with(warm_start_ratio=0.35)),
        ("RLS final con δ=0.05 (poca reg.)",           "rls",    "rls",    RLS_M,    rls_with(delta=0.05)),
    ])

    abl = []
    for name, fam, algo, M, params in variants:
        t0 = time.perf_counter()
        r = eval_filter(algo, M, params, subset, y_sub)
        abl.append({"Variante": name, "fam": fam, **r, "s": time.perf_counter() - t0})
        print(f"{name:<48} AAE={r['aae']:5.2f}  MedAE={r['medae']:4.2f}  Out={r['out']:4.1f}%  ({abl[-1]['s']:4.1f}s)")
    abl = pd.DataFrame(abl)

    # Compromiso de dilatación a apertura fija: acondicionamiento vs. tiempo de cómputo
    print(f"\n--- Compromiso de dilatación a apertura fija (~{T_AP_SAMPLES * 8.0:.0f} ms) ---")
    dil_rows = []
    for d_tr in (1, 2, 3, 4):
        m_tr = max(6, min(MAX_M, int(round(T_AP_SAMPLES / d_tr)) + 1))
        cond_tr = []
        for a in ac_sub[:100]:
            Ut = build_U(a, M=m_tr, delay_step=d_tr); et = np.linalg.eigvalsh(Ut.T @ Ut)
            cond_tr.append(et[-1] / max(et[0], 1e-12))
        t0 = time.perf_counter()
        r_tr = eval_filter("rls", m_tr, rls_with(delay_step=d_tr), subset, y_sub)
        elapsed_ms = 1000.0 * (time.perf_counter() - t0) / len(y_sub)
        dil_rows.append({
            "D": d_tr, "M": m_tr, "L=3M": 3 * m_tr,
            "Apertura_ms": round((m_tr - 1) * d_tr * 8.0, 1),
            "cond_mediano": float(np.median(cond_tr)),
            "AAE": round(r_tr["aae"], 2), "MedAE": round(r_tr["medae"], 2),
            "Outliers_%": round(r_tr["out"], 1), "ms_ventana": round(elapsed_ms, 2),
        })
    display(pd.DataFrame(dil_rows).set_index("D"))

    order = abl.sort_values("aae", ascending=False)
    fig, ax = plt.subplots(figsize=(11, 0.45 * len(order) + 1.5))
    cols = [COLORS["rls"] if f == "rls" else COLORS["nlms"] for f in order["fam"]]
    ax.barh(order["Variante"], order["aae"], color=cols)
    for y_, v in enumerate(order["aae"]):
        ax.text(v, y_, f" {v:.2f}", va="center", fontsize=9)
    ax.axvline(base_err.mean(), color="#7f7f7f", ls="--", label=f"Sin filtro ({base_err.mean():.2f})")
    ax.set_xlabel("AAE (BPM) sobre 300 ventanas de entrenamiento")
    ax.set_title("Ablación por componente (azul = familia LMS, rojo = RLS)")
    ax.legend(loc="lower right"); ax.grid(axis="y", visible=False)
    fig.tight_layout()
    save_fig(fig, "07_ablacion")
    plt.show()

if RUN_SENSITIVITY:
    neff_opt = float(RLS_PARAMS.get("N_eff", 1409.3))
    neffs = sorted(set([150, 300, 600, 1000, int(round(neff_opt)), 2500])) + [np.inf]
    delta_opt_val = float(RLS_PARAMS["delta"])
    deltas = sorted(set([0.05, 0.2, 1.0, 5.0, 15.0, 30.0, 100.0, 500.0, round(delta_opt_val, 3)]))

    def run_sens(over):
        p = dict(RLS_PARAMS)
        for k, v in over.items():
            if v is None: p.pop(k, None)
            else: p[k] = v
        return eval_filter("rls", RLS_M, p, subset, y_sub)

    r_neff = [run_sens({"N_eff": None, "lam": 1.0} if np.isinf(n) else {"N_eff": float(n)}) for n in neffs]
    r_delta = [run_sens({"delta": d}) for d in deltas]

    print("\nSensibilidad a N_eff:", [(n, round(r["aae"], 2), round(r["medae"], 2)) for n, r in zip(neffs, r_neff)])
    print("Sensibilidad a delta:", [(d, round(r["aae"], 2), round(r["medae"], 2)) for d, r in zip(deltas, r_delta)])

    fig, axes = plt.subplots(1, 2, figsize=(14.5, 4.4))
    xs = np.arange(len(neffs))
    axes[0].plot(xs, [r["aae"] for r in r_neff], "o-", color=COLORS["rls"], label="AAE")
    axes[0].plot(xs, [r["medae"] for r in r_neff], "s--", color="#ff9896", label="Error mediano")
    axes[0].set_xticks(xs); axes[0].set_xticklabels([("∞ (λ=1)" if np.isinf(n) else f"{n:g}") for n in neffs])
    axes[0].set_xlabel("Memoria efectiva N_eff (muestras)"); axes[0].set_ylabel("BPM")
    axes[0].set_title("Sensibilidad a N_eff (λ = 1 − 1/N_eff)"); axes[0].legend()

    axes[1].plot(deltas, [r["aae"] for r in r_delta], "o-", color=COLORS["rls"], label="AAE")
    axes[1].plot(deltas, [r["medae"] for r in r_delta], "s--", color="#ff9896", label="Error mediano")
    axes[1].axvline(delta_opt_val, color="k", ls=":", lw=1.1, label=f"Óptimo CV (δ={delta_opt_val:.2f})")
    axes[1].set_xscale("log")
    axes[1].set_xlabel("δ (regularización inicial, P[-1] = 1/(δ·L))"); axes[1].set_ylabel("BPM")
    axes[1].set_title("Sensibilidad a δ (compromiso sesgo-varianza)"); axes[1].legend()

    fig.tight_layout()
    save_fig(fig, "08_sensibilidad_RLS")
    plt.show()

## 8. Control de calidad de señal (SQI) y regímenes de movimiento

Criterio dual previo al filtrado (**compuerta AND**, sin usar etiquetas ni depender del algoritmo): una ventana se marca como irrecuperable si su energía dinámica de acelerómetro supera el percentil $75$ del entrenamiento ($E_{\text{ACC}}>P_{75}^{\text{train}}$) **y** su SNR espectral de pico previa al filtrado es inferior al percentil $25$ del entrenamiento ($\text{SNR}_{\text{pico}}<P_{25}^{\text{train}}$).

In [ ]:
def calcular_metricas_sqi(X, fs=FS):
    X = np.asarray(X, dtype=float)
    N = len(X)
    acc_ac = X[:, 2:5, :] - np.mean(X[:, 2:5, :], axis=-1, keepdims=True)
    e_acc = np.mean(np.sum(acc_ac ** 2, axis=1), axis=-1)
    p1, p2, _ = preprocess_signal(X, fs=fs)
    snr_db = np.empty(N)
    for i in range(N):
        d = snr_weighted_late_fusion(p1[i], p2[i], fs=fs)
        snr_db[i] = 10.0 * np.log10(estimate_cardiac_snr(d, fs=fs) + 1e-12)
    return {"e_acc": e_acc, "snr_db": snr_db}


def calibrar_umbrales_sqi(met_train, pct_acc=SQI_PCT_ACC, pct_snr=SQI_PCT_SNR):
    return {
        "pct_acc": pct_acc, "pct_snr": pct_snr,
        "umbral_acc": float(np.nanpercentile(met_train["e_acc"], pct_acc)),
        "umbral_snr_db": float(np.nanpercentile(met_train["snr_db"], pct_snr)),
    }


def evaluar_criterio_dual_sqi(met, umbrales):
    cond_acc_alta = met["e_acc"] > umbrales["umbral_acc"]
    cond_snr_baja = met["snr_db"] < umbrales["umbral_snr_db"]
    no_finito = ~np.isfinite(met["e_acc"]) | ~np.isfinite(met["snr_db"])
    mask_irr = (cond_acc_alta & cond_snr_baja) | no_finito
    mask_val = ~mask_irr
    return {
        "total": len(mask_val),
        "solo_acc_alta": int(np.sum(cond_acc_alta & ~cond_snr_baja)),
        "solo_snr_baja": int(np.sum(~cond_acc_alta & cond_snr_baja)),
        "ambos_descartadas": int(np.sum(mask_irr)),
        "retenidas": int(np.sum(mask_val)),
        "umbral_acc": umbrales["umbral_acc"],
        "umbral_snr_db": umbrales["umbral_snr_db"],
        "e_acc": met["e_acc"], "snr_db": met["snr_db"],
        "cond_acc_alta": cond_acc_alta, "cond_snr_baja": cond_snr_baja,
        "mask_irrecuperable": mask_irr, "mask_valida": mask_val,
    }


met_train = calcular_metricas_sqi(X_train)
met_test = calcular_metricas_sqi(X_test)
umbrales_sqi = calibrar_umbrales_sqi(met_train)
diag_sqi = evaluar_criterio_dual_sqi(met_test, umbrales_sqi)

pcts = [5, 25, 50, 75, 95]
print("Percentiles SNR de pico (dB) [5, 25, 50, 75, 95]:")
print("  Train:", np.round(np.nanpercentile(met_train["snr_db"], pcts), 2))
print("  Test :", np.round(np.nanpercentile(met_test["snr_db"], pcts), 2))
print(f"Umbral ACC (P{SQI_PCT_ACC} train): {diag_sqi['umbral_acc']:.4f} | Umbral SNR (P{SQI_PCT_SNR} train): {diag_sqi['umbral_snr_db']:.2f} dB")
print(f"Descartadas AND: {diag_sqi['ambos_descartadas']}/{diag_sqi['total']} ({100*diag_sqi['ambos_descartadas']/diag_sqi['total']:.2f}%) | Retenidas: {diag_sqi['retenidas']}")

# --- Figura 09: Diagnóstico SQI ---
m_val, m_irr = diag_sqi["mask_valida"], diag_sqi["mask_irrecuperable"]
fig, axes = plt.subplots(1, 3, figsize=(19, 5.0))
ex = np.maximum(diag_sqi["e_acc"], 1e-6)
axes[0].scatter(ex[m_val], diag_sqi["snr_db"][m_val], c="#1f77b4", alpha=0.5, s=18, label=f"Válidas ({m_val.sum()})")
axes[0].scatter(ex[m_irr], diag_sqi["snr_db"][m_irr], c="#d62728", marker="x", s=40, linewidths=1.4, label=f"Descartadas AND ({m_irr.sum()})")
axes[0].axvline(diag_sqi["umbral_acc"], color="black", ls="--", lw=1.2, label=f"Umbral E_ACC (P{SQI_PCT_ACC} train)")
axes[0].axhline(diag_sqi["umbral_snr_db"], color="darkorange", ls="--", lw=1.2, label=f"Umbral SNR (P{SQI_PCT_SNR} train)")
axes[0].set_xscale("log"); axes[0].set_xlabel("Energía dinámica del acelerómetro (log)"); axes[0].set_ylabel("SNR de pico de la PPG (dB)")
axes[0].set_title("Mapa SQI (cuadrante inferior derecho = descarte)"); axes[0].legend(loc="best", fontsize=8)

bins = np.linspace(np.nanmin([met_train["snr_db"].min(), met_test["snr_db"].min()]),
                   np.nanmax([met_train["snr_db"].max(), met_test["snr_db"].max()]), 40)
axes[1].hist(met_train["snr_db"], bins=bins, alpha=0.55, density=True, color="#7f7f7f", label="Train")
axes[1].hist(met_test["snr_db"], bins=bins, alpha=0.55, density=True, color="#1f77b4", label="Test")
axes[1].axvline(diag_sqi["umbral_snr_db"], color="darkorange", ls="--", lw=1.5, label="Umbral SNR")
axes[1].set_xlabel("SNR de pico (dB)"); axes[1].set_ylabel("Densidad"); axes[1].set_title("Distribución de la SNR de pico"); axes[1].legend(fontsize=8.5)

irr_s = np.array([100.0 * np.mean(m_irr[test_groups == s]) for s in subjects])
solo_s = np.array([100.0 * np.mean(diag_sqi["cond_acc_alta"][test_groups == s] & m_val[test_groups == s]) for s in subjects])
xp = np.arange(len(subjects))
axes[2].bar(xp, irr_s, color="#d62728", alpha=0.85, label="Descartadas (ACC alta AND SNR baja)")
axes[2].bar(xp, solo_s, bottom=irr_s, color="#ff7f0e", alpha=0.75, label="Solo ACC alta (retenidas)")
axes[2].set_xticks(xp); axes[2].set_xticklabels([f"S{int(s)+1}" for s in subjects])
axes[2].set_ylabel("% de ventanas del sujeto"); axes[2].set_title("Descarte por sujeto"); axes[2].legend(fontsize=8.5)
fig.tight_layout(); save_fig(fig, "09_diagnostico_sqi"); plt.show()

# --- Figura 10: Ejemplos de ventana retenida vs descartada ---
snr_db = diag_sqi["snr_db"]
cand_val = np.where(diag_sqi["cond_acc_alta"] & m_val)[0]
cand_irr = np.where(m_irr)[0]
idx_val_ex = int(cand_val[np.nanargmax(snr_db[cand_val])])
idx_irr_ex = int(cand_irr[np.nanargmin(snr_db[cand_irr])])
fig, axes = plt.subplots(2, 2, figsize=(15, 6.5))
freqs_plot = np.fft.rfftfreq(2048, d=1.0 / FS) * 60.0
for col, (idx, titulo) in enumerate([
    (idx_val_ex, f"Ventana #{idx_val_ex}: ACC alta pero retenida (SNR={snr_db[idx_val_ex]:.1f} dB)"),
    (idx_irr_ex, f"Ventana #{idx_irr_ex}: Descartada (SNR={snr_db[idx_irr_ex]:.1f} dB)"),
]):
    ppg_ex = X_test[idx, 0, :] - np.mean(X_test[idx, 0, :])
    acc_mag = np.linalg.norm(X_test[idx, 2:5, :] - np.mean(X_test[idx, 2:5, :], axis=-1, keepdims=True), axis=0)
    axes[0, col].plot(t_axis, ppg_ex / (np.std(ppg_ex) + 1e-9), label="PPG1 (norm)", color="#1f77b4", lw=1.1)
    axes[0, col].plot(t_axis, acc_mag / (np.std(acc_mag) + 1e-9) - 3.5, label="|ACC| (norm, despl.)", color="#7f7f7f", lw=1.0, alpha=0.8)
    axes[0, col].set_title(titulo, fontweight="bold", fontsize=10); axes[0, col].set_xlabel("Tiempo (s)"); axes[0, col].set_ylabel("Amplitud normalizada"); axes[0, col].legend(loc="upper right", fontsize=8)
    psd_ppg = np.abs(np.fft.rfft(ppg_ex * np.hanning(WIN), n=2048)) ** 2
    psd_acc = np.abs(np.fft.rfft((acc_mag - acc_mag.mean()) * np.hanning(WIN), n=2048)) ** 2
    mb = (freqs_plot >= 30) & (freqs_plot <= 240)
    axes[1, col].plot(freqs_plot[mb], psd_ppg[mb] / np.max(psd_ppg[mb]), label="PSD PPG1", color="#1f77b4", lw=1.3)
    axes[1, col].plot(freqs_plot[mb], psd_acc[mb] / np.max(psd_acc[mb]), label="PSD |ACC|", color="#ff7f0e", lw=1.1, ls="--")
    axes[1, col].axvline(y_test[idx], color="#2ca02c", lw=2, label=f"ECG real ({y_test[idx]:.1f} BPM)")
    axes[1, col].set_xlabel("Frecuencia (BPM)"); axes[1, col].set_ylabel("Densidad espectral norm."); axes[1, col].legend(loc="upper right", fontsize=8)
fig.tight_layout(); save_fig(fig, "10_ejemplos_sqi"); plt.show()

# --- Impacto SQI, control aleatorio y sensibilidad de umbrales ---
def control_aleatorio(err, mask_valida, n_draw=N_RANDOM_CTRL, seed=SEED):
    rng = np.random.default_rng(seed)
    N = len(err); n_disc = int((~mask_valida).sum()); aae_sqi = float(err[mask_valida].mean())
    total = err.sum(); draws = np.empty(n_draw)
    for b in range(n_draw):
        disc = rng.choice(N, size=n_disc, replace=False)
        draws[b] = (total - err[disc].sum()) / (N - n_disc)
    return aae_sqi, float(draws.mean()), float(np.mean(draws <= aae_sqi))

sqi_rows = []
for k in ORDER:
    m_all = compute_metrics(y_test, test_eval[k]["preds"])
    m_v = compute_metrics(y_test, test_eval[k]["preds"], mask=m_val)
    m_i = compute_metrics(y_test, test_eval[k]["preds"], mask=m_irr)
    _, aae_r, p_emp = control_aleatorio(test_eval[k]["abs_err"], m_val) if k != "base" else (0.0, np.nan, np.nan)
    sqi_rows.append({
        "Método": LABELS[k],
        "Global": round(m_all["aae"], 2), "Retenidas": round(m_v["aae"], 2),
        "Descartadas": round(m_i["aae"], 2), "Aleatorio": round(aae_r, 2) if np.isfinite(aae_r) else np.nan,
        "p_emp": round(p_emp, 3) if np.isfinite(p_emp) else np.nan,
    })
display(pd.DataFrame(sqi_rows).set_index("Método"))

sens_sqi = []
err_n, err_r = test_eval["nlms"]["abs_err"], test_eval["rls"]["abs_err"]
for p_acc in [65, 75, 85]:
    for p_snr in [15, 25, 35]:
        d_t = evaluar_criterio_dual_sqi(met_test, calibrar_umbrales_sqi(met_train, p_acc, p_snr))
        v = d_t["mask_valida"]
        sens_sqi.append({"P_ACC": f"P{p_acc}", "P_SNR": f"P{p_snr}", "Descart_%": round(100*(~v).mean(), 2),
                         "NLMS": round(safe_mean(err_n[v]), 2), "RLS": round(safe_mean(err_r[v]), 2),
                         "Mejora": round(safe_mean(err_n[v]) - safe_mean(err_r[v]), 2)})
display(pd.DataFrame(sens_sqi))

# --- Figura 11: Regímenes de movimiento y ECDF post-SQI ---
q33, q66 = np.percentile(diag_sqi["e_acc"], [33.3, 66.7])
regimenes = [
    ("Reposo / Bajo\n(P0–P33)", diag_sqi["e_acc"] <= q33),
    ("Moderado\n(P33–P66)", (diag_sqi["e_acc"] > q33) & (diag_sqi["e_acc"] <= q66)),
    ("Intenso (Todas)\n(P66–P100)", diag_sqi["e_acc"] > q66),
    ("Intenso Válidas\n(P66–P100 + SQI)", (diag_sqi["e_acc"] > q66) & m_val),
]
fig, axes = plt.subplots(1, 2, figsize=(14.5, 5.0))
x_reg = np.arange(len(regimenes)); w = 0.20
reg_table = []
for j, k in enumerate(["base", "lms", "nlms", "rls"]):
    aaes = [float(np.mean(np.abs(y_test[m] - test_eval[k]["preds"][m]))) for _, m in regimenes]
    axes[0].bar(x_reg + (j - 1.5) * w, aaes, width=w, color=COLORS[k], label=LABELS[k])
    reg_table.append({"Método": LABELS[k], **{r[0].replace("\n", " "): round(v, 2) for r, v in zip(regimenes, aaes)}})
axes[0].set_xticks(x_reg); axes[0].set_xticklabels([r[0] for r in regimenes]); axes[0].set_ylabel("AAE (BPM)")
axes[0].set_title("Degradación por Intensidad de Movimiento y Efecto del SQI", fontweight="bold"); axes[0].legend(fontsize=8.5)

for k in ORDER:
    es = np.sort(np.abs(y_test[m_val] - test_eval[k]["preds"][m_val]))
    axes[1].plot(es, np.arange(1, len(es)+1)/len(es)*100.0, color=COLORS[k], lw=2.0 if k=="rls" else 1.4, label=LABELS[k])
axes[1].axvline(OUTLIER_BPM, color="k", ls="--", lw=1.0, label=f"Umbral Outlier ({OUTLIER_BPM:.0f} BPM)")
axes[1].set_xlim(0, 35); axes[1].set_xlabel("Error Absoluto (BPM)"); axes[1].set_ylabel("% Acumulado de Ventanas Válidas (SQI)")
axes[1].set_title("Distribución Acumulada del Error (ECDF) en Ventanas Válidas", fontweight="bold"); axes[1].legend(loc="lower right", fontsize=8.5)
fig.tight_layout(); save_fig(fig, "11_regimenes_movimiento_y_ecdf"); plt.show()

print("AAE por régimen de movimiento:")
display(pd.DataFrame(reg_table).set_index("Método"))

## 9. Resumen cuantitativo final

In [ ]:
best_lms = min(["lms", "nlms", "vs_lms"], key=lambda k: test_eval[k]["aae"])
b, r_ = test_eval[best_lms], test_eval["rls"]
print("=" * 78)
print("RESUMEN (test, detector propio de cada método)")
print("=" * 78)
print(f"Mejor variante LMS                : {LABELS[best_lms]}  (AAE {b['aae']:.2f}, MedAE {b['medae']:.2f}, outliers {b['outlier_pct']:.2f}%)")
print(f"RLS (Proyecto 2)                  : RLS   (AAE {r_['aae']:.2f}, MedAE {r_['medae']:.2f}, outliers {r_['outlier_pct']:.2f}%)")
print(f"Reducción relativa de AAE         : {100 * (b['aae'] - r_['aae']) / b['aae']:.1f}%")
print(f"Reducción de outliers             : {b['outlier_pct'] - r_['outlier_pct']:.2f} puntos porcentuales")
print(f"Con detector común (Hann 8192)    : {LABELS[best_lms]} {b['common']['aae']:.2f} vs RLS {r_['common']['aae']:.2f} (AAE)")
print(f"Tiempo por ventana (filtro+fusión): {LABELS[best_lms]} {times[best_lms]:.1f} ms vs RLS {times['rls']:.1f} ms")
print(f"Figuras guardadas en              : {os.path.abspath(FIG_DIR)}")